# M5 PySpark Flight Delay Practice — VS Code Runnable Version

This version is fixed for **VS Code local Jupyter**.

Important: Your error came from using **Python 3.13** with PySpark.  
Use **Python 3.11** for this notebook.

Recommended setup:

```bash
conda create -n pyspark311 python=3.11 -y
conda activate pyspark311
pip install pyspark==3.5.1 pandas notebook ipykernel
python -m ipykernel install --user --name pyspark311 --display-name "Python 3.11 PySpark"
```

Then open this notebook in VS Code and select kernel:

**Python 3.11 PySpark**

In [ ]:
import os
import sys
from pathlib import Path

print("Python version:", sys.version)

if sys.version_info >= (3, 13):
    raise RuntimeError(
        "This notebook should not run with Python 3.13. "
        "Please switch VS Code kernel to Python 3.11 PySpark."
    )

# Set a safe local Spark temp directory
os.environ["SPARK_LOCAL_DIRS"] = str(Path("./spark_tmp").resolve())
Path("./spark_tmp").mkdir(exist_ok=True)

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, when, count

spark = (
    SparkSession.builder
    .appName("M5_FlightDelay_VSCode")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Spark session created successfully")

## Local Data Path

Put your flight delay CSV file in the local `data/` folder.

Expected example path:

```text
data/flights.csv
```

If your file has a different name, update `csv_path` below.

In [ ]:
from pathlib import Path

csv_path = Path("data/flights.csv")

if not csv_path.exists():
    print("Data file not found:", csv_path.resolve())
    print("Please create a data folder and place your flight delay CSV file there.")
else:
    df = spark.read.csv(str(csv_path), header=True, inferSchema=True)
    print("Dataset loaded successfully")
    df.printSchema()
    df.show(5)

# VS Code Runnable PySpark Notebook

This notebook is converted to run locally in VS Code.

Install packages first:

```bash
pip install pyspark pandas
```


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySparkPracticeVSCode") \
    .getOrCreate()

print("Spark session created successfully")

##**Step 1**: Install Spark

##**Step 2**: Setting up Spark

Before you can connect to a Spark cluster, Spark needs to be installed. The code below is boilerplate code that can be used to set-up Spark. Please note that this code will be leveraged in all the notebooks since each nodebook is a separate entity.

## **Step 3**. Import the lib

## **Step 4**: Using DataFrames

Spark's core data structure is the Resilient Distributed Dataset (RDD), which is a low-level object designed to handle large datasets by distributing data across multiple nodes in a cluster. However, working directly with RDDs can be challenging. To simplify this, Spark provides DataFrames, which offer a higher-level abstraction built on top of RDDs.

A Spark DataFrame functions similarly to a SQL table, where columns represent attributes and rows represent observations. DataFrames not only make it easier to manipulate and analyze data, but they are also more optimized for complex operations compared to RDDs.

Read the Airlines.csv file from your Google Drive into a Spark DataFrame

Output the column names and the "rows" and "column" counts

In [ ]:
#outputing the column names
df.columns #Column Names

In [ ]:
#Outputing the row count
df.count()  #Row Count
#Output the column count
len(df.columns) #Column Count

In [ ]:
df.dtypes

In order to examine the summary of any particular column of a DataFrame, we use the describe method.

The describe "method" gives us the statistical summary of the given column

In [ ]:
#Examine the summary of the "DepDelay" column
df.describe('DepDelay').show()

In order to select particular columns from the DataFrame, use the select method

In [ ]:
df.select('ArrDelay','DepDelay').show()

Selecting Distinct Multiple Columns

In [ ]:
df.select('ArrDelay','DepDelay').distinct().show()

## **Step 5.** Filtering Data

In [ ]:
df.filter(df.Origin=='SFO').show()

Filtering Data (Multiple Parameters)

In [ ]:
df.filter((df.Origin=='SFO')&(df.Dest=='PDX')).show()

## **Step 6**: Performing SQL Queries
- SQL queries can be passes directly to any DataFrame; for that, we need to create a table from the DataFrame using the registerTempTable

In [ ]:
# old (deprecated)
# df.registerTempTable('airlines')

# new (Spark 2.0+)
df.createOrReplaceTempView("airlines")

Typically the entry point into all SQL functionality in Spark is the SQLContext class. To create a basic instance of this call, all we need is a SparkContext reference.

In [ ]:
from pyspark.sql import SQLContext
sqlContext = SQLContext(spark)
sqlContext

In [ ]:
sqlContext.sql('select * from airlines').show()

In [ ]:
sqlContext.sql('select distinct(Dest) from airlines').show(20)

In [ ]:
print(df.count())

In [ ]:
print(df.take(3))

In [ ]:
## Filter#3
spark.sql("""SELECT DISTINCT Origin, Dest, Distance
FROM airlines WHERE distance > 1000
ORDER BY distance DESC""").show(10)

## **Step 7**: Aggregation

In [ ]:
## aggregation
spark.sql("""SELECT concat(Origin, Dest) as market,
avg(Distance)
FROM airlines
GROUP BY 1
ORDER BY 1 DESC""").show(10)

In [ ]:
## aggregation
spark.sql("""SELECT concat(Origin, Dest) as market,
min(Distance)
FROM airlines
GROUP BY 1
ORDER BY 1 DESC""").show(10)

In [ ]:
## Top 10
## "SFOSMF" 86 actually is "860" due to the bugs of pyspark show function
spark.sql("""SELECT concat(Origin, Dest) as market,
max(Distance)
FROM airlines
GROUP BY 1
ORDER BY max(Distance) DESC
limit 10""").show(10)

# Student Practice

In [ ]:
# Practice 1
# Display first 5 rows

# TODO:
# df.show(5)

In [ ]:
# Practice 2
# Count total records

# TODO:
# df.count()

In [ ]:
# Practice 3
# Calculate average delay by Origin

# TODO:
# Use groupBy() and avg()

## Student Practice Answers

In [ ]:
# Answer 1
df.show(5)

In [ ]:
# Answer 2
df.count()

In [ ]:
# Answer 3
from pyspark.sql.functions import avg

df.groupBy("Origin").agg(
    avg("ArrDelay").alias("avg_delay")
).show()

## Student Practice

In [ ]:
# Practice 1
# Show the first 10 rows and print the schema.

# TODO:
# df.show(10)
# df.printSchema()

In [ ]:
# Practice 2
# Count total records in the dataset.

# TODO:
# df.count()

In [ ]:
# Practice 3
# Calculate average arrival delay by Origin airport.

# TODO:
# from pyspark.sql.functions import avg
# df.groupBy("Origin").agg(avg("ArrDelay").alias("avg_delay")).show()

In [ ]:
# Practice 4
# Create a delay_flag column.
# If ArrDelay > 15, label as "Delayed"; otherwise "OnTime".

# TODO:
# from pyspark.sql.functions import when, col
# df = df.withColumn("delay_flag", when(col("ArrDelay") > 15, "Delayed").otherwise("OnTime"))
# df.show(5)

## Student Practice Answers

In [ ]:
# Answer 1
df.show(10)
df.printSchema()

In [ ]:
# Answer 2
total_records = df.count()
print("Total records:", total_records)

In [ ]:
# Answer 3
df.groupBy("Origin").agg(
    avg("ArrDelay").alias("avg_delay")
).orderBy(col("avg_delay").desc()).show()

In [ ]:
# Answer 4
df = df.withColumn(
    "delay_flag",
    when(col("ArrDelay") > 15, "Delayed").otherwise("OnTime")
)

df.select("Origin", "Dest", "ArrDelay", "delay_flag").show(10)

## Stop Spark Session

In [ ]:
spark.stop()
print("Spark stopped")